In [1]:
import random
import numpy as np
import torch
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast
# from exp.exp_imputation import Exp_Imputation
# from exp.exp_short_term_forecasting import Exp_Short_Term_Forecast
# from exp.exp_anomaly_detection import Exp_Anomaly_Detection
# from exp.exp_classification import Exp_Classification
# from data_provider.data_creat import *
# import akshare as ak
from torch.utils.tensorboard import SummaryWriter

/root/projects/Time-Series-Library/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Args:
    '''基本配置'''
    # 选项：[long_term_forecast, short_term_forecast, imputation, classification, anomaly_detection]')
    task_name = 'long_term_forecast'
    model_id = 'stock_cy'
    des = '20240107' # 实验描述 20231210
    # 模型名称，选项：[Autoformer, Transformer, TimesNet]
    model = 'TimeXer'

    '''数据加载'''
    # 数据集类型,选项：[ETTh1,ETTh2,ETTm1,ETTm2,custom,m4,PSM,MSL,SMAP,SMD,SWAT,UEA]
    data = 'custom'
    test_path = 'raw_stock_cy_300798.csv'
    root_path = './dataset/Stock/'
    data_path = '2024_01_07_14_06_07stock_cy_all_raw.csv'
    # 预测任务 M:多变量预测多变量, S:单变量预测单变量, MS:多变量预测单变量
    features = 'MS'
    # 目标列名，S或MS任务中的目标特征
    target = 'OT'
    # 时间采集粒度，选项：[s:秒, t:分钟, h:小时, d:天, b:工作日, w:周, m:月]
    freq = 'd'
    # 模型检查点的位置
    checkpoints = './checkpoints/'

    '''是否训练模型'''
    is_training = 1 # 设置为1则进行训练，设置为0则进行测试
    # 测试集的比例
    if is_training:
        test_ratio = [0.1,0.1] # test测试集、valid验证集的占比
        pred_once = False # 是否只预测一次，用于测试集数据
    else:
        test_ratio = [0.6,0.1]
        data_path = test_path
        pred_once = True # 是否只预测一次，用于测试集数据
    scale = True # dataset获取的源数据是否进行归一化
    back_training = True # 是否从上次训练的模型继续训练

    '''预测任务
    预测的基本思想是将已知序列延长到（seq_len+pred_len），即预测后的总长度。然后通过几个 TimesBlock 层以及层归一化，表示一些潜在的周期内和周期间信息。利用这些信息，我们可以将其投影到输出空间。然后通过 denorm（如果是非平稳 Transformer）我们得到最终的输出。
    '''
    # 还原预测数据为原始数据
    inverse = scale
    # 输入序列长度,这是用于模型训练的输入序列的长度
    seq_len = 20
    # 开始标记长度,这是模型输出目标中有标签数据的长度，类似于滑动窗口的长度
    label_len = 5
    # 预测序列长度
    pred_len = 1
    # 季节模式（针对M4数据集）
    seasonal_patterns = 'Monthly'


    '''插补任务'''
    # 插补任务中数据丢失率
    mask_rate = 0.25

    '''异常检测任务'''
    # 异常检测中异常点占比
    anomaly_ratio = 0.25

    '''模型定义'''
    # 模型维度，模型中所有子层（包括自注意力层）的输出维度，设置在256至512之间，数字越大显存用的越多
    d_model = 64
    # 前馈网络（FFN）的维度，每个注意力层之后提供额外的表达能力，d_ff是d_model的两到四倍。
    d_ff = 256
    # TimesBlock 中傅里叶变换,频率排名前k个周期
    top_k = 5
    # Inception 中卷积核个数，用于 Inception
    num_kernels = 6
    # encoder 输入特征数
    enc_in = 19
    # decoder 输入特征数
    dec_in = enc_in
    # 输出通道数
    c_out = enc_in
    # 多头注意力机制
    n_heads = 8
    # encoder 层数，编码器层数
    e_layers = 2
    # decoder 层数，解码器层数
    d_layers = 1
    # 滑动窗口长度
    moving_avg = 20
    # 对 Q 进行采样，注意力因子数量
    factor = 3
    # 是否下采样操作 pooling，也就是对模型进行压缩
    distil = False
    # dropout 率
    dropout = 0.1
    # 时间特征嵌入方式,选项：[timeF, fixed, learned]
    embed = 'timeF'
    # 激活函数类型
    activation = 'gelu'
    # 是否输出 attention，也就是是否输出注意力权重
    output_attention = False
    # 是否使用归一化（Non-stationary Transformer 归一化），1 是 0 否
    use_norm = 1
    # patch 长度（TimeXer/PatchTST 等模型使用），需能整除 seq_len
    patch_len = 5

    '''优化'''
    # 并行核心数
    num_workers = 10
    # 实验轮数
    itr = 1
    # 训练迭代次数
    train_epochs = 500
    # batch size 大小
    batch_size = 256
    # early stopping 机制容忍次数
    patience = 3
    # 学习率
    learning_rate = 0.0001
    # 损失函数
    loss = 'MSE'
    # 学习率下降策略
    lradj = 'type1'
    # 使用混合精度训练
    use_amp = False

    '''GPU'''
    # 使用 gpu
    use_gpu = True
    gpu = 0
    # 使用多个 gpus
    use_multi_gpu = False
    # 多 gpu 的设备 id
    devices = '0,1,2,3'

    '''去平稳化投影仪参数'''
    # 投影仪的隐藏层维度（列表）
    p_hidden_dims = [128, 128]
    # 投影仪中的隐藏层数
    p_hidden_layers = 2


# 创建参数对象
args = Args()

# 设置随机种子以确保结果可重现
fix_seed = 2021
random.seed(fix_seed)
torch.manual_seed(fix_seed)
np.random.seed(fix_seed)

In [3]:
# 检查并设置 GPU
args.use_gpu = torch.cuda.is_available() and args.use_gpu
if args.use_gpu:
    print("使用 GPU.")
    total_cuda_devices = torch.cuda.device_count()  # 获取系统中可用的 GPU 总数
    print(f"系统中总共有 {total_cuda_devices} 个 CUDA 设备可用。")
    if args.use_multi_gpu:
        args.devices = args.devices.replace(' ', '')
        device_ids = args.devices.split(',')
        args.device_ids = [int(id_) for id_ in device_ids]
        args.gpu = args.device_ids[0]

        # 打印多 GPU 使用情况
        print(f"使用多个GPU: {args.device_ids}")
        device = torch.device(f"cuda:{args.gpu}" if args.use_gpu else "cpu")
        print(f"Primary GPU (cuda:{args.gpu}) is in use.")
    else:
        args.gpu = 0
        device = torch.device("cuda" if args.use_gpu else "cpu")
        print("使用单个 GPU.")
else:
    device = torch.device("cpu")
    print("使用 CPU.")

# 选择合适的实验类
if args.task_name == 'long_term_forecast':
    Exp = Exp_Long_Term_Forecast
# elif args.task_name == 'short_term_forecast':
#     Exp = Exp_Short_Term_Forecast
# elif args.task_name == 'imputation':
#     Exp = Exp_Imputation
# elif args.task_name == 'anomaly_detection':
#     Exp = Exp_Anomaly_Detection
# elif args.task_name == 'classification':
#     Exp = Exp_Classification
else:
    Exp = Exp_Long_Term_Forecast  # 默认情况

# 进行训练和测试
if args.is_training:
    # 多次训练模型
    for ii in range(args.itr):
        setting = '{}_{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}'.format(
            args.task_name, # 任务名称
            args.model_id, # 模型id
            args.model, # 模型名称
            args.data, # 数据集名称
            args.features, # 预测任务
            args.seq_len, # 输入序列长度
            args.label_len, # 开始标记长度
            args.pred_len, # 预测序列长度
            args.d_model, # encoder 输入特征数
            args.n_heads, # 多头注意力机制
            args.e_layers, # encoder 层数
            args.d_layers, # decoder 层数
            args.d_ff, # FFN 层隐含神经元个数
            args.factor, # 对 Q 采样的因子数
            args.embed, # 时间特征嵌入方式
            args.distil, # 是否下采样操作 pooling
            args.des,  # 实验描述
            ii) # 实验轮数

        exp = Exp(args) # 创建实验对象
        print('>>>>>>>开始训练 : {}>>>>>>>>>>>>>>>>>>>>>>>>>>'.format(setting))
        exp.train(setting) # 训练模型

        print('>>>>>>>测试 : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
        exp.test(setting) # 测试模型
        if args.use_gpu:
            torch.cuda.empty_cache()
else:
    # 模型测试
    ii = 0
    setting = '{}_{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}'.format(
        args.task_name,
        args.model_id,
        args.model,
        args.data,
        args.features,
        args.seq_len,
        args.label_len,
        args.pred_len,
        args.d_model,
        args.n_heads,
        args.e_layers,
        args.d_layers,
        args.d_ff,
        args.factor,
        args.embed,
        args.distil,
        args.des,
        ii)
    exp = Exp(args)  # 创建实验对象
    print('>>>>>>>测试 : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
    exp.test(setting, test=1) # 测试模型
    if args.use_gpu:
        torch.cuda.empty_cache()

使用 GPU.
系统中总共有 1 个 CUDA 设备可用。
使用单个 GPU.
Use GPU: cuda:0
🚀 Lazy Loading: TimeXer ...
>>>>>>>开始训练 : long_term_forecast_stock_cy_TimeXer_custom_ftMS_sl20_ll5_pl1_dm64_nh8_el2_dl1_df256_fc3_ebtimeF_dtFalse_20240107_0>>>>>>>>>>>>>>>>>>>>>>>>>>


ValueError: BuilderConfig '2024_01_07_14_06_07stock_cy_all_raw' not found. Available: ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'electricity', 'traffic', 'weather', 'exchange_rate', 'national_illness', 'm4-yearly', 'm4-quarterly', 'm4-monthly', 'm4-weekly', 'm4-daily', 'm4-hourly', 'EthanolConcentration', 'FaceDetection', 'Handwriting', 'Heartbeat', 'JapaneseVowels', 'PEMS-SF', 'SelfRegulationSCP1', 'SelfRegulationSCP2', 'SpokenArabicDigits', 'UWaveGestureLibrary', 'SMD-data', 'SMD-label', 'MSL-data', 'MSL-label', 'SMAP-data', 'SMAP-label', 'PSM-data', 'PSM-label', 'SWaT']

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir=./tensorboard_logs/